In [ ]:
# Library set up
import pandas as pd
import numpy as np

import seaborn as sns
from matplotlib import pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), ".."))
if root_path not in sys.path:
    sys.path.append(root_path)

from sklearn.pipeline import Pipeline
from src.features.engineer import create_preprocessor, FeatureEngineer
from src.preprocess.preprocessor import preprocess

import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)


#### **1. Loading Raw Data**

In [ ]:
df = pd.read_csv("../data/raw/train.csv")
print(df.shape)
df.head(3)

#### **2. Duplication and Missing Values**

This is a generated data so there is no logical erros, missing values or duplication, but for real-world preparation, `clean_logical_error()`, `drop.na()` and `drop_duplicates()` are still incorporated in the preprocessing


In [ ]:
def clean_logical_errors(df):
    initial_shape = df.shape[0]
    
    # 1. Sanity check (True means retain, False means drop)
    # Age: range from 18 to 100
    cond_age = (df['Age'] >= 18) & (df['Age'] <= 100)
    
    # Numerical values >= 0
    cond_tenure = df['Tenure'] >= 0
    cond_usage = df['Usage Frequency'] >= 0
    cond_calls = df['Support Calls'] >= 0
    cond_delay = df['Payment Delay'] >= 0
    cond_spend = df['Total Spend'] >= 0
    
    # Last Interaction: can not negative or larger than a year (365 days)
    cond_interaction = (df['Last Interaction'] >= 0) & (df['Last Interaction'] <= 365)

    # 2. Conditions
    clean_df = df[
        cond_age & 
        cond_tenure & 
        cond_usage & 
        cond_calls & 
        cond_delay & 
        cond_spend & 
        cond_interaction
    ].copy()
    
    # 3. Result
    dropped_rows = initial_shape - clean_df.shape[0]
    print(f"Remove {dropped_rows} unlogical row(s)")
    
    return clean_df

In [ ]:
df = df.dropna()
df = df.drop_duplicates()
df = clean_logical_errors(df)
print('Data shape after Dropping', df.shape)

#### **3. Feature Selection & Justification**

| Dropped Variable     | Reasons for Removal |
|---------------------|-------------------|
| CustomerID          | A unique identifier with no predictive power; inclusion would lead to overfitting on specific IDs. |
| Tenure              | Very low correlation with target (Churn ≈ -0.05); provides negligible predictive signal. |
| Usage Frequency     | Near-zero correlation with Churn (≈ -0.05); contributes little to model performance and may add noise. |
| Subscription Type   | Extremely low correlation with Churn (≈ -0.02); minimal information gain for classification. |
| Contract Length | Although the 'Monthly' category (~20% of the data) exhibits a 100% churn rate, this pattern is likely an artifact of the synthetic data generation process. The remaining categories (~80%) show no clear class separation, resulting in ~ 0 correlation with the target. |

In [ ]:
print('Data shape before Dropping', df.shape)
df = df.drop(
    ["CustomerID", "Tenure", "Usage Frequency", "Subscription Type", "Contract Length"], axis=1
)

df = df.dropna()
print('Data shape after Dropping', df.shape)
df.head()

#### **3. Ensuring Data Type**

In [ ]:
# int 
df['Churn'] = df['Churn'].astype(int)
df['Age'] = df['Age'].astype(int)
df['Support Calls'] = df['Support Calls'].astype(int) # Number of calls
df['Payment Delay'] = df['Payment Delay'].astype(int) # Days 
df['Last Interaction'] = df['Last Interaction'].astype(int) # Days ago

# float
df['Total Spend'] = df['Total Spend'].astype(float) # Currency Units

# Category
df['Gender'] = df['Gender'].astype('str')

df.info()

#### **4. OHE Gender**

Transform `Gender` into Binary feature `Male`, where `Male == 0` means customer is Female and `Male == 1` means otherwise


In [ ]:
gender_ohe = pd.get_dummies(df["Gender"], dtype=np.int8)
df = df.drop(["Gender"], axis=1)
df = df.join(gender_ohe)
df = df.drop(columns= 'Female')
df.head(3)

#### **5. Binning Numerical Features**

#### **5.1. Age Groups**

**Binning Age Groups Jusstification**

The continuous variable `Age` was transformed into categorical `Age Group` segments to capture non-linear relationships with the target variable. 

As we discuss in EDA Part, churn rates vary meaningfully across age brackets, with the `Senior` group (>60) exhibiting a significantly higher churn rate (100%) compared to other groups (~48–57%). 

This transformation allows the model to better distinguish behavioral patterns across different life stages, which may not be effectively captured by a linear relationship with age. Additionally, grouping improves interpretability and reduces sensitivity to minor variations in age values.

**Encoder Decision**:

One-hot encoding was used because churn does not follow a monotonic trend across age groups (e.g., 25–39 has lower churn than 18–24 and 40–59), so ordinal encoding would impose a misleading order.

In [ ]:
def classify_age_group(age):
    if 18 <= age <= 24:
        return "Young Adult"
    elif 24 < age <= 39:
        return "Adult"
    elif 39 < age <= 59:
        return "Mid-Career"
    elif age >= 60:
        return "Senior"


df["Age_Group"] = df["Age"].apply(classify_age_group)
df.head(3)

In [ ]:
# Quick vizualization

age_order = ["Young Adult", "Adult", "Mid-Career", "Senior"]
df["Age_Group"] = pd.Categorical(df["Age_Group"], categories=age_order, ordered=True)
# --------------------------------------------------

# 2. Prepare data 
age_churn_rate = df.groupby('Age_Group', observed=True)['Churn'].mean().reset_index()
age_churn_rate['Churn_Percentage'] = age_churn_rate['Churn'] * 100

# Khi tính count, sort_index() bây giờ sẽ dựa trên thứ tự categorical thay vì chữ cái
age_dist = df['Age_Group'].value_counts().sort_index().reset_index()
age_dist.columns = ['Age_Group', 'Count']


# 3. Plot
sns.set_style("white")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Subplot 1: Distribution ---
sns.barplot(data=age_dist, x='Age_Group', y='Count', ax=axes[0], color="#a1c9f4")

axes[0].set_title('Age Group Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age Group')
axes[0].set_ylabel('Count')


# --- Subplot 2: Churn Rate ---
sns.barplot(data=age_churn_rate, x='Age_Group', y='Churn_Percentage',
            ax=axes[1], color="#4d99d0")

axes[1].set_title('Churn Rate by Age Group', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Age Group')
axes[1].set_ylabel('Churn Rate (%)')

# Annotate %
for p in axes[1].patches:
    height = p.get_height()
    if height > 0:
        axes[1].annotate(f'{height:.1f}%',
                         (p.get_x() + p.get_width() / 2., height),
                         ha='center', va='bottom',
                         fontsize=9, fontweight='bold',
                         xytext=(0, 4),
                         textcoords='offset points')

plt.tight_layout()
plt.show()

In [ ]:
# OHE encoder

encoder = OneHotEncoder(
    categories=[["Young Adult", "Adult", "Mid-Career", "Senior"]],  drop="first", handle_unknown="ignore",
    sparse_output=False   
)

# Fit & transform
age_ohe = encoder.fit_transform(df[['Age_Group']])

age_ohe_df = pd.DataFrame(
    age_ohe, columns=encoder.get_feature_names_out(['Age_Group']),
    index=df.index
)

df = pd.concat([df, age_ohe_df], axis=1)

# drop Age_Group
df.drop(columns=['Age_Group'], inplace=True)

df.head(3)


#### **5.2. Last Interaction**

**Binning Age Groups Jusstification**

The continuous variable `Last Interaction` was discretized into categorical segments to capture non-linear relationships between engagement recency and churn risk.

Results from EDA indicate a clear threshold effect: 

- Customers with more than 15 days since their last interaction exhibit a significantly higher churn rate (~66.5%) compared to those within 15 days (~49%). 

- While the distinction between the 0–7 and 8–15 day groups is minimal, this segmentation was retained for interpretability and consistency with common engagement-based customer grouping.

Given the synthetic nature of the dataset, these thresholds should be interpreted as heuristic rather than definitive, but they provide a practical abstraction for modeling customer inactivity risk.

**Encoder Decision**:

One-hot encoding was preferred since “Highly Active” and “Active” show nearly identical churn rates (~49%), while “Dormant” is significantly higher (~66.5%), making ordinal relationships inappropriate.

In [ ]:
def classify_interaction_frequency(last_interaction):
    if 0 < last_interaction <= 7:
        return "Highly Active"
    elif 7 < last_interaction <= 15:
        return "Active"
    elif last_interaction > 15:
        return "Dormant"


df["Interaction Frequency"] = df["Last Interaction"].apply(
    classify_interaction_frequency
)
df.head()

In [ ]:
# Quick Vizualization

order = ["Highly Active", "Active", "Dormant"]

# --- Color mapping (consistent) ---
color_map = {
    "Highly Active": "#a1c9f4",  # light blue
    "Active": "#6fa8dc",         # medium blue
    "Dormant": "#1f4e79"         # dark blue (highlight risk)
}

colors = [color_map[o] for o in order]

# --- Churn rate ---
churn_rate_by_group = df.groupby("Interaction Frequency")["Churn"].mean().reindex(order).reset_index()
churn_rate_by_group["Churn Rate (%)"] = churn_rate_by_group["Churn"] * 100

# --- Distribution ---
dist = df["Interaction Frequency"].value_counts().reindex(order)

# --- Plot ---
sns.set_style("white")
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# --- Subplot 1: Pie chart ---
axes[0].pie(
    dist,
    labels=order,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors
)
axes[0].set_title("Interaction Frequency Distribution", fontsize=13, fontweight='bold')

# --- Subplot 2: Bar chart ---
ax = sns.barplot(
    data=churn_rate_by_group,
    x="Interaction Frequency",
    y="Churn Rate (%)",
    ax=axes[1],
    palette=color_map,
    order=order
)

# Annotate %
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.annotate(f'{height:.1f}%',
                    (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom',
                    fontsize=11,
                    xytext=(0, 5),
                    textcoords='offset points')

axes[1].set_title("Churn Rate by Interaction Frequency", fontsize=13, fontweight='bold')
axes[1].set_ylabel("Churn Rate (%)")
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# OHE encoder

encoder = OneHotEncoder(
    categories=[["Highly Active", "Active", "Dormant"]],  drop="first", handle_unknown="ignore",
    sparse_output=False   
)

# Fit & transform
interact_ohe = encoder.fit_transform(df[["Interaction Frequency"]])

interact_ohe_df = pd.DataFrame(
    interact_ohe, columns=encoder.get_feature_names_out(["Interaction Frequency"]),
    index=df.index
)

df = pd.concat([df, interact_ohe_df], axis=1)

# drop Age_Group
df.drop(columns=["Interaction Frequency"], inplace=True)

df.head(3)


#### **6. Scaler (fit for Train dataset only)**

Standardization was applied to numerical features (`Age`, `Support Calls`, `Payment Delay`, `Total Spend`, and `Last Interaction`) to address differences in scale and measurement units. 

These variables exhibit heterogeneous ranges: `Age` represents demographic information, `Support Calls` and `Payment Delay` capture discrete behavioral counts, `Total Spend` reflects monetary value with potentially large variance, and `Last Interaction` measures recency in days. 

Without scaling, features with larger magnitudes—particularly `Total Spend`—could dominate the learning process, leading to biased model coefficients. By applying StandardScaler, all features are transformed to zero mean and unit variance, ensuring balanced contribution and improving model convergence and stability.

In [ ]:
# When traning model, we will split X,y and Train, test first and apply scale for X_train only
drop_cols = ["Churn",'Male', 'Age_Group_Adult', 'Age_Group_Mid-Career', 
             'Age_Group_Senior', 'Interaction Frequency_Active','Interaction Frequency_Dormant']
scaler = StandardScaler()
df_no_label = df.drop(columns= drop_cols)
df_no_label = pd.DataFrame(
    scaler.fit_transform(df_no_label), columns=df_no_label.columns
)
df_no_label.head()

In [ ]:
df = df_no_label.join(df[drop_cols])
df.head()

#### **7. Rerunning with Pipeline**

In [ ]:
# 1. raw data
df = pd.read_csv("../data/raw/train.csv")

## preprocessing
df = preprocess(df=df)

target = "Churn"

X = df.drop(columns=[target])
y = df[target]

# 2. train, test split and Shuffle
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

In [ ]:
# 3. Feature engineering Pipeline
preprocesing_pipeline = Pipeline([
   ("feature_engineering", FeatureEngineer()), # binning Age and Last Interaction
   ("preprocessor", create_preprocessor()) # Scaler and OHE
])

X_train_transformed = preprocesing_pipeline.fit_transform(X_train)

In [ ]:
feature_names = preprocesing_pipeline.named_steps["preprocessor"].get_feature_names_out()

X_train_df = pd.DataFrame(X_train_transformed, columns=feature_names)
X_train_df.head()